## Summary

This notebook evaluates how DME is associated with cumulative 305-day milk yield within the bi-modal milk flow framework.
It loads the relevant lactation data, prepares analysis variables, and summarizes the key model and chart outputs for interpretation.

# Bi-modal milk flow project

Analysis of the effect of DME on cumulative 305-day milk yield


In [0]:
%pip install --upgrade shap "numpy<=2.1"
%restart_python
import shap
shap.initjs()
print(shap.__version__)

In [0]:
#import packages 
import pandas as pd 
import numpy as np 
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import seaborn as sns
# from statsmodels.formula.api import ols
from scipy.stats import norm 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit

Load data from blob 

In [0]:
#read dataframe from the US blob 
parquet_file_path = "wasbs://bovianalytics-data@bovianalyticsstorage.blob.core.windows.net/us/SunnySide/lactation_curve_characteristics"

# Read the Parquet file into a DataFrame > now set to a limit to first make the code working
bi_modal_df = spark.read.parquet(parquet_file_path)

## Descriptive stats

In [0]:
#Descriptive stats df
number_of_recordings = bi_modal_df.count()
number_of_animals = bi_modal_df.select('AnimalNumber').distinct().count()
number_of_parities = bi_modal_df.select('LactationNumber').distinct().count()
number_of_lactations = bi_modal_df.select('LactationId').distinct().count()
print(f'Number of recordings: {number_of_recordings}')
print(f'Number of animals: {number_of_animals}')
print(f'Number of parities: {number_of_parities}')
print(f'Number of lactations: {number_of_lactations}')
print(f'Average amount of lactations per animal: {number_of_lactations/number_of_animals}')


In [0]:
#mean time to peak 
print('Mean time to peak: ', bi_modal_df.select(F.avg('TimeToPeak')).collect()[0][0])
print('Mean peak milk yield', bi_modal_df.select(F.avg('PeakYield')).collect()[0][0])
print('Mean 305 milk yield', bi_modal_df.select(F.avg('CummulativeYield')).collect()[0][0])
#US average ~ 11000 kg
print('Mean persistency', bi_modal_df.select(F.avg('Persistency')).collect()[0][0])



In [0]:
#convert to panda to make data easier to plot
pdf = bi_modal_df.toPandas()

In [0]:
#visualize model variables 
plt.hist(pdf['TimeToPeak'], bins=100)
plt.title('Time to peak distribution')
plt.xlabel('Time to peak')
plt.ylabel('Frequency')
plt.show()

plt.hist(pdf['PeakYield'], bins=100)
plt.title('Peak distribution')
plt.xlabel('Peak')
plt.ylabel('Frequency')
plt.show()

plt.hist(pdf['CummulativeYield'], bins=100)
plt.title('Cummulative yield distribution')
plt.xlabel('Cummulative yield')
plt.ylabel('Frequency')
plt.show()

plt.hist(pdf['Persistency'], bins=100)
plt.title('Persistency distribution')
plt.xlabel('Persistency')
plt.ylabel('Frequency')
plt.show()

plt.hist(pdf['DailyMilkYield'], bins = 100)
plt.title('Daily milk yield distribution')
plt.xlabel('Daily milk yield')
plt.ylabel('Frequency')
plt.show

### Create categorial variable for DME 

In [0]:
#alternative create a categorial variable out of BMI prop 
#rename column BIM to DME 
pdf.rename(columns={'BIM_prop': 'DME_prop'}, inplace=True)
pdf.rename(columns ={'BIM_prop_logit': 'DME_prop_logit'}, inplace=True)

# #equal split in three
# pdf["DME_prop_cat"] = pd.qcut(pdf["DME_prop"], q=3, labels=["low", "middle", "high"])

#split based on a threshold 
bins = [0, 0.20, 0.40, 1.0]
labels = ["low", "middle", "high"]

pdf["DME_prop_cat"] = pd.cut(pdf["DME_prop"], bins=bins, labels=labels, include_lowest=True)


### Create categorial variable for lactation number

In [0]:
#convert lactationnumber to 4 categories, 1,2,3,4 where everything above 4 becomes 4. 
pdf["LactationCat"] = pdf["LactationNumber"].clip(upper=4)

### Descriptive stats cummulative milk yield 

In [0]:
#plot cummulative milk yield vs DME prop and vs Logit DME

plt.scatter(pdf['CummulativeYield'], pdf['DME_prop'])
# add trendline 
lr = LinearRegression()
lr.fit(pdf['CummulativeYield'].values.reshape(-1, 1), pdf['DME_prop'])
plt.plot(pdf['CummulativeYield'], lr.predict(pdf['CummulativeYield'].values.reshape(-1, 1)), color='red')
plt.title('Cummulative yield vs DME prop')
plt.xlabel('Cummulative yield')
plt.ylabel('DME prop')
plt.show()  

plt.scatter(pdf['CummulativeYield'], pdf['DME_prop_logit'])
# add trendline 
lr = LinearRegression()
lr.fit(pdf['CummulativeYield'].values.reshape(-1, 1), pdf['DME_prop_logit'])
plt.plot(pdf['CummulativeYield'], lr.predict(pdf['CummulativeYield'].values.reshape(-1, 1)), color='red')
plt.title('Cummulative yield vs Logit DME')
plt.xlabel('Cummulative yield')
plt.ylabel('Logit BMI')
plt.show()


In [0]:
#plot cummulative milk yield vs LactationNumber, CalvingSeason and CalvingYear 

plt.scatter(pdf['CummulativeYield'], pdf['LactationNumber'])
# add trendline 
lr = LinearRegression()
lr.fit(pdf['CummulativeYield'].values.reshape(-1, 1), pdf['LactationNumber'])
plt.plot(pdf['CummulativeYield'], lr.predict(pdf['CummulativeYield'].values.reshape(-1, 1)), color='red')
plt.title('Cummulative yield vs lactation number')
plt.xlabel('Cummulative yield')
plt.ylabel('Lactation number')
plt.show()  

# Boxplot: CummulativeYield by CalvingSeason
data = [pdf.loc[pdf['CalvingSeason'] == season, 'CummulativeYield'] for season in sorted(pdf['CalvingSeason'].unique())]
plt.boxplot(data, labels=sorted(pdf['CalvingSeason'].unique()))
plt.title('Cummulative yield vs CalvingSeason')
plt.xlabel('CalvingSeason')
plt.ylabel('Cummulative yield')
plt.show()

# Boxplot: CummulativeYield by CalvingYear
data = [
    pdf.loc[pdf['CalvingYear'] == year, 'CummulativeYield']
    for year in sorted(pdf['CalvingYear'].unique())
]
plt.boxplot(data, labels=sorted(pdf['CalvingYear'].unique()))
plt.title('Cummulative yield vs CalvingYear')
plt.xlabel('CalvingYear')
plt.ylabel('Cummulative yield')
plt.show()

#### Correlations

In [0]:
#test if predictors are correlated 
# Convert CalvingSeason to numeric codes if it is categorical
pdf['CalvingSeason_cat'] = pdf['CalvingSeason'].astype('category').cat.codes

# Compute correlation matrix
corr_matrix = pdf[
    ['DME_prop_logit', 'CalvingSeason_cat', 'LactationNumber', 'CalvingYear' , 'CummulativeYield']
].corr()

sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm'
)
plt.title('Correlation Matrix')
plt.show()

Also shows neg correlation between DME_prop_logit and lactationnumber 
neg correlation between DME and cummulativemilkyield 

## Train-test split

In [0]:
#perform a train test split of the dataset based on AnimalNumber
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(pdf, groups=pdf["AnimalNumber"])
)

train_df = pdf.iloc[train_idx]
test_df = pdf.iloc[test_idx] #now contains 21.2 % of the data 


#for the train set
#convert calvingseason by one hot encoding 
train_df = pd.get_dummies(train_df, columns=['CalvingSeason'], prefix = 'CalvingSeason', dtype=int)
#rename columns so they match again with the seasons 
train_df.rename(columns={'CalvingSeason_0': 'CalvingSeason_Spring', 'CalvingSeason_1': 'CalvingSeason_Summer', 'CalvingSeason_2': 'CalvingSeason_Autumn', 'CalvingSeason_3': 'CalvingSeason_Winter'}, inplace=True)


#mean calvingyear train set 
mean_calving_year = train_df['CalvingYear'].mean()
#center calvingyear 
train_df['CalvingYear'] = train_df['CalvingYear'] - mean_calving_year

#for the test set 
#convert calvingseason by one hot encoding 
test_df = pd.get_dummies(test_df, columns=['CalvingSeason'], prefix = 'CalvingSeason', dtype=int)
#rename columns so they match again with the seasons 
test_df.rename(columns={'CalvingSeason_0': 'CalvingSeason_Spring', 'CalvingSeason_1': 'CalvingSeason_Summer', 'CalvingSeason_2': 'CalvingSeason_Autumn', 'CalvingSeason_3': 'CalvingSeason_Winter'}, inplace=True)

#check for data leakage by checking if the AnimalNumbers are truely seperated
assert set(train_df["AnimalNumber"]).isdisjoint(test_df["AnimalNumber"])


## Linear Regression models

### Classic

In [0]:
# fit linear regression model 


#drop winter 
model_305 = ols('CummulativeYield ~ DME_prop_logit +C(LactationCat) + CalvingYear + CalvingSeason_Spring + CalvingSeason_Summer + CalvingSeason_Autumn', data=train_df)
model_305 = model_305.fit()
print(model_305.summary())

#visualize results
y = train_df['CummulativeYield'].values
yhat = model_305.fittedvalues

plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)  # 1:1 line

plt.xlabel("Observed Cummulative milk yield")
plt.ylabel("Fitted Cummulative milk yield")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

In [0]:
# Create visualizations to check model assumptions
plt.figure(figsize=(12, 6))

# qq-plot to see if residuals are normally distributed
plt.subplot(1, 2, 1)

residuals = model_305.resid
# Step 1: Fit a normal distribution to the residuals
mu, sigma = norm.fit(residuals)

# Step 2: Sort the residuals
sorted_residuals = np.sort(residuals)

# Step 3: Compute the theoretical quantiles from the normal distribution
n = len(residuals)
# create equally spaced probabilities (avoiding 0 and 1)
probabilities = (np.arange(1, n + 1) - 0.5) / n
theoretical_quantiles = norm.ppf(probabilities, loc=mu, scale=sigma)

# Step 4: Plot
plt.scatter(theoretical_quantiles, sorted_residuals, color='blue', label='Residuals')
plt.plot(theoretical_quantiles, theoretical_quantiles, color='red', lw=2, label='45° line')
plt.xlabel('Theoretical Quantiles (Normal)')
plt.ylabel('Sample Quantiles (Residuals)')
plt.title('QQ plot of residuals')
plt.legend()


# residuals vs fitted values to check for homoscedasticity
plt.subplot(1, 2, 2)
plt.scatter(yhat, model_305.resid)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted values')

plt.tight_layout()
plt.show()

#### Coverting LactationNumber to categorial variables

In [0]:
# Inverse logit function
def logit(x, eps=1e-6):
    return np.log((x + eps) / (1 - x + eps))

# Range of proportions
DME_vals = np.linspace(0, 1, 100)

# Coefficient for DME_prop_logit
beta = model_305.params['DME_prop_logit']

# Δ milk yield relative to DME_prop = 0
delta_yield = beta * (logit(DME_vals) - logit(0.000001))

plt.figure(figsize=(8, 5))
plt.plot(DME_vals, delta_yield)
plt.xlabel("DME proportion")
plt.ylabel("Δ Cummulative Yield relative to no DME")
plt.title("Effect of DME proportion on Cumulative Milk Yield")
plt.grid(True)
plt.show()

Transforming Logit(BMI) back to BMI 

In [0]:
# take a reference row 
df_pred = train_df.iloc[[0]].copy()

# duplicate it so we have two rows
df_pred = pd.concat([df_pred]*2, ignore_index=True)

# set DME proportions
df_pred.loc[0, "DME_prop"] = 0.000001
df_pred.loc[1, "DME_prop"] = 0.5

# recompute transformed variable
df_pred["DME_prop_logit"] = logit(df_pred["DME_prop"])

# predictions
pred = model_305.predict(df_pred)

reduction_05 = pred.iloc[1] - pred.iloc[0]
print(reduction_05)


In [0]:
beta = model_305_cont.params["DME_prop_logit"]

def effect_of_1pct(p):
    eps = 1e-6
    logit = lambda x: np.log((x+eps) / (1-x+eps))
    return beta * (logit(p + 0.01) - logit(p))

# Example: compute effect at p = 0.20 (20% DMEodal)
for p in [0.05, 0.10, 0.20, 0.30, 0.40]:
    print(f"At DME_prop={p:.2f}, +1 percentage point changes milk yield by {effect_of_1pct(p):.4f}")
    
mean_p = train_df["DME_prop"].mean()
effect_mean = effect_of_1pct(mean_p)
print("Effect at mean DME_prop:", effect_mean)  

##### Eval test set

In [0]:
#Evaluate using the testset 
#center calvingyear using the mean of the train set 
test_df['CalvingYear'] = test_df['CalvingYear'] - mean_calving_year

# Use the full test_df and let the model handle categorical encoding
y_test = test_df['CummulativeYield'].values
ypred_test = model_305_cont.predict(test_df)
print('Test R2: ', r2_score(y_test, ypred_test))
print('Test RMSE: ', np.sqrt(mean_squared_error(y_test, ypred_test)))

#visualize results

plt.figure(figsize=(6,6))
plt.scatter(y_test, ypred_test, alpha=0.5, color = 'darkorange')
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2, color = 'darkorange')  # 1:1 line

plt.xlabel("Observed Cummulative milk yield")
plt.ylabel("Fitted Cummulative milk yield")
plt.title("Observed vs Fitted TEST set")
plt.tight_layout()
plt.show()

#### BMI as category

In [0]:
#DME
#drop winter 
model_305_cat = ols('CummulativeYield ~ C(DME_prop_cat) + C(LactationCat) + CalvingYear + CalvingSeason_Spring + CalvingSeason_Summer + CalvingSeason_Autumn', data=train_df)
model_305_cat = model_305_cat.fit()
print(model_305_cat.summary())

#visualize results
y = train_df['CummulativeYield'].values
yhat = model_305_cat.fittedvalues

plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)  # 1:1 line

plt.xlabel("Observed Cummulative milk yield")
plt.ylabel("Fitted Cummulative milk yield")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

##### Eval test set

In [0]:
#Evaluate using the testset 
#center calvingyear using the mean of the train set 
# test_df['CalvingYear'] = test_df['CalvingYear'] - mean_calving_year

y_test = test_df['CummulativeYield'].values
ypred_test = model_305_cat.predict(test_df)
print('Test R2: ', r2_score(y_test, ypred_test))
print('Test RMSE: ', np.sqrt(mean_squared_error(y_test, ypred_test)))

#visualize results

plt.figure(figsize=(6,6))
plt.scatter(y_test, ypred_test, alpha=0.5, color = 'darkorange')
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2, color = 'darkorange')  # 1:1 line

plt.xlabel("Observed Cummulative milk yield")
plt.ylabel("Fitted Cummulative milk yield")
plt.title("Observed vs Fitted TEST set")
plt.tight_layout()
plt.show()

In [0]:
# fit linear regression model 

#drop winter 
model_305 = ols('CummulativeYield ~ C(DME_prop_cat) + C(LactationCat) + CalvingYear + CalvingSeason_Spring + CalvingSeason_Summer + CalvingSeason_Autumn', data=train_df)
model_305 = model_305.fit()
print(model_305.summary())

#visualize results
y = train_df['CummulativeYield'].values
yhat = model_305.fittedvalues

plt.figure(figsize=(6,6))
plt.scatter(y, yhat, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2)  # 1:1 line

plt.xlabel("Observed Cummulative milk yield")
plt.ylabel("Fitted Cummulative milk yield")
plt.title("Observed vs Fitted")
plt.tight_layout()
plt.show()

print('Train RMSE: ', np.sqrt(mean_squared_error(y, yhat)))

In [0]:
y_test = test_df['CummulativeYield'].values
ypred_test = model_305.predict(test_df)
print('Test R2: ', r2_score(y_test, ypred_test))
print('Test RMSE: ', np.sqrt(mean_squared_error(y_test, ypred_test)))

#visualize results

plt.figure(figsize=(6,6))
plt.scatter(y_test, ypred_test, alpha=0.5, color = 'darkorange')
plt.plot([y.min(), y.max()], [y.min(), y.max()], linewidth=2, color = 'darkorange')  # 1:1 line

plt.xlabel("Observed Cummulative milk yield")
plt.ylabel("Fitted Cummulative milk yield")
plt.title("Observed vs Fitted TEST set")
plt.tight_layout()
plt.show()

In [0]:
grouped = pdf.groupby("DME_prop_cat")["CummulativeYield"]

mean = grouped.mean()
sem = grouped.sem()                 # standard error of the mean
ci95 = 1.96 * sem                   # 95% CI

plt.figure(figsize=(6,4))
plt.bar(mean.index, mean.values, yerr=ci95.values, capsize=5)
plt.ylabel("Mean cumulative milk yield (kg)")
plt.xlabel("DME proportion category")
plt.title("Cumulative milk yield by DME proportion category\n(Mean ± 95% CI)")
plt.tight_layout()
plt.show()


In [0]:
grouped = pdf.groupby("DME_prop_cat")["CummulativeYield"]

mean = grouped.mean()
sem = grouped.sem()                 # standard error of the mean
ci95 = 1.96 * sem                   # 95% CI

# -----------------------------
# Broken y-axis configuration
# -----------------------------
low_ylim = (0, 10500)               # lower part (context)
high_ylim = (11500, 13500)          # focus region (important part)

fig = plt.figure(figsize=(6, 5))
plt.tight_layout()

# GridSpec to control height ratios (70% focus region)
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.05)

ax_high = fig.add_subplot(gs[0])
ax_low = fig.add_subplot(gs[1], sharex=ax_high)

# -----------------------------
# Plot bars on both axes
# -----------------------------
ax_high.bar(mean.index, mean.values, yerr=ci95.values, capsize=5)
ax_low.bar(mean.index, mean.values, yerr=ci95.values, capsize=5)

# -----------------------------
# Set y-limits
# -----------------------------
ax_high.set_ylim(high_ylim)
ax_low.set_ylim(low_ylim)

# -----------------------------
# Remove spines between axes
# -----------------------------
ax_high.spines['bottom'].set_visible(False)
ax_low.spines['top'].set_visible(False)
ax_high.tick_params(labeltop=False)
ax_low.xaxis.tick_bottom()

# -----------------------------
# Diagonal break marks
# -----------------------------
d = .015
kwargs = dict(transform=ax_high.transAxes, color='k', clip_on=False)
ax_high.plot((-d, +d), (-d, +d), **kwargs)
ax_high.plot((1 - d, 1 + d), (-d, +d), **kwargs)

kwargs = dict(transform=ax_low.transAxes, color='k', clip_on=False)
ax_low.plot((-d, +d), (1 - d, 1 + d), **kwargs)
ax_low.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

# -----------------------------
# Labels and title
# -----------------------------
ax_low.set_xlabel("DME proportion category")
ax_high.set_ylabel("Mean cumulative milk yield (kg)")
fig.suptitle("Cumulative milk yield by DME proportion category\n(Mean ± 95% CI)", y=0.95)

plt.tight_layout()
plt.show()


In [0]:
plt.figure(figsize=(6,4))
pdf.boxplot(
    column="CummulativeYield",
    by="DME_prop_cat",
    grid=False
)
plt.ylabel("Cumulative milk yield (kg)")
plt.xlabel("DME proportion category")
plt.title("Distribution of cumulative milk yield by DME category")
plt.suptitle("")  # remove automatic pandas title
plt.tight_layout()
plt.show()

## Random Forest

In [0]:
#define input and output variables 
X_train = train_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_train = train_df['CummulativeYield']
X_test = test_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_test = test_df['CummulativeYield']

#define parameter grid 
param_grid = {
    "n_estimators": [150, 300, 500, 800, 1000],
    "max_depth": [3, 5, 7, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5]
    }

#define model
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=10,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
best_rf_model = grid_search.best_estimator_

# Predictions
y_pred_train = best_rf_model.predict(X_train)

#calculate training metrics 
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
r_squared_train = r2_score(y_train, y_pred_train)
print('Training RMSE: ', rmse_train)
print('Training R2: ', r_squared_train)

#feature importance 
feature_names = ['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']

importances = pd.Series(best_rf_model.feature_importances_, index=feature_names)
importances.sort_values().plot.barh(title = 'Feature importance RF model')
plt.show()



### Eval test set

In [0]:
y_pred_test = best_rf_model.predict(X_test)

#calculate testing metrics 
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r_squared_test = r2_score(y_test, y_pred_test)
print('Testing RMSE: ', rmse_test)
print('Testing R2: ', r_squared_test)

## Random forrest with normal DME prop

In [0]:
#redefine input and output variables 
X_train = train_df[['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_train = train_df['CummulativeYield']
X_test = test_df[['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_test = test_df['CummulativeYield']

#define parameter grid 
param_grid = {
    "n_estimators": [250, 500, 1000],
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5]
    }

#define model
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=10,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
print("Best parameters:", grid_search.best_params_)
best_rf_model = grid_search.best_estimator_

# Predictions
y_pred_train = best_rf_model.predict(X_train)

#calculate training metrics 
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
r_squared_train = r2_score(y_train, y_pred_train)
print('Training RMSE: ', rmse_train)
print('Training R2: ', r_squared_train)

#feature importance 
feature_names = ['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']

importances = pd.Series(best_rf_model.feature_importances_, index=feature_names)
importances.sort_values().plot.barh(title = 'Feature importance RF model non logit')
plt.show()


In [0]:
y_pred_test = best_rf_model.predict(X_test)

#calculate testing metrics 
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r_squared_test = r2_score(y_test, y_pred_test)
print('non logit')
print('Testing RMSE: ', rmse_test)
print('Testing R2: ', r_squared_test)

## XgBoost model with hyperparameter tuning

In [0]:
#define input and output variables 
X_train = train_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_train = train_df['CummulativeYield']
X_test = test_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_test = test_df['CummulativeYield']

#Grid search parameters
param_grid = {
    "n_estimators": [250, 500, 1000],
    "max_depth": [1, 3, 5, 7],
    "learning_rate": [0.001, 0.01, 0.1],
    "min_child_weight": [1, 3, 5]
    }

    
#define model
grid_search = GridSearchCV(
        estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
        param_grid=param_grid,
        cv=10,
        n_jobs=-1
        )

 # Fit model
grid_search.fit(X_train, y_train) 
print("Best parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_

# Predictions
y_pred = best_model.predict(X_train)

# RMSE
rmse = np.sqrt(mean_squared_error(y_train, y_pred))
print(f"RMSE: {rmse:.3f}")

r2 = r2_score(y_train, y_pred)
print(f"R²: {r2:.3f}")

# Feature importance 
importance_XgBoost = pd.Series(best_model.feature_importances_, index=feature_names)
importance_XgBoost = importance_XgBoost.sort_values(ascending=True)

importance_XgBoost.plot.barh(figsize=(6,4))
plt.title(f"Feature Importance XgBoost logit")
plt.show()

### Eval test set

In [0]:
y_pred_test = best_model.predict(X_test)

#calculate testing metrics 
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r_squared_test = r2_score(y_test, y_pred_test)
print('Testing RMSE: ', rmse_test)
print('Testing R2: ', r_squared_test)

## XgBoost non logit

In [0]:
#define input and output variables 
X_train = train_df[['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_train = train_df['CummulativeYield']
X_test = test_df[['DME_prop', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn']]
y_test = test_df['CummulativeYield']


#Grid search parameters
param_grid = {
    "n_estimators": [250, 500, 1000],
    "max_depth": [1, 3, 5, 7],
    "learning_rate": [0.001, 0.01, 0.1],
    "min_child_weight": [1, 3, 5]
    }

    
#define model
grid_search = GridSearchCV(
        estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
        param_grid=param_grid,
        cv=10,
        n_jobs=-1
        )

 # Fit model
grid_search.fit(X_train, y_train) 
print("Best parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_

# Predictions
y_pred = best_model.predict(X_train)

# RMSE
rmse = np.sqrt(mean_squared_error(y_train, y_pred))
print(f"RMSE: {rmse:.3f}")

r2 = r2_score(y_train, y_pred)
print(f"R²: {r2:.3f}")

# Feature importance 
importance_XgBoost = pd.Series(best_model.feature_importances_, index=feature_names)
importance_XgBoost = importance_XgBoost.sort_values(ascending=True)

importance_XgBoost.plot.barh(figsize=(6,4))
plt.title(f"Feature Importance XgBoost non logit")
plt.show()

In [0]:
y_pred_test = best_model.predict(X_test)

#calculate testing metrics 
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r_squared_test = r2_score(y_test, y_pred_test)
print('non logit')
print('Testing RMSE: ', rmse_test)
print('Testing R2: ', r_squared_test)

### Permutation feature importance 

In [0]:
#recreate best fitted models for RF and XgBoost 
#define input and output variables 
X_train = train_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_train = train_df['CummulativeYield']
X_test = test_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_test = test_df['CummulativeYield']

#fit model 
RF = RandomForestRegressor(n_estimators=1000, max_depth=3, min_samples_leaf=5, min_samples_split=2, random_state=42).fit(X_train, y_train)
XgBoost = XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators = 1000, learning_rate = 0.01, min_child_weight = 1, max_depth = 1 ).fit(X_train, y_train)

# Predictions
rf_y_pred_train = RF.predict(X_train)
x_y_pred_train = XgBoost.predict(X_train)

#calculate training metrics 
rmse_train = np.sqrt(mean_squared_error(y_train, rf_y_pred_train))
r_squared_train = r2_score(y_train, rf_y_pred_train)
print('Training RMSE RF: ', rmse_train)
print('Training R2 RF: ', r_squared_train)
rmse_train = np.sqrt(mean_squared_error(y_train, x_y_pred_train))
r_squared_train = r2_score(y_train, x_y_pred_train)
print('Training RMSE XgBoost: ', rmse_train)
print('Training R2 XgBoost: ', r_squared_train)


from sklearn.inspection import permutation_importance

# ---- Permutation Importance for Random Forest ----
rf_perm = permutation_importance(
    RF, 
    X_test, 
    y_test, 
    n_repeats=30,     
    random_state=42, 
    scoring='r2'
)

rf_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': rf_perm.importances_mean,
    'importance_std': rf_perm.importances_std
}).sort_values(by='importance_mean', ascending=False)

print("Random Forest - Permutation Importance")
print(rf_importance)


# ---- Permutation Importance for XGBoost ----
xgb_perm = permutation_importance(
    XgBoost, 
    X_test, 
    y_test, 
    n_repeats=30, 
    random_state=42, 
    scoring='r2'
)

xgb_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': xgb_perm.importances_mean,
    'importance_std': xgb_perm.importances_std
}).sort_values(by='importance_mean', ascending=False)

print("\nXGBoost - Permutation Importance")
print(xgb_importance)



In [0]:
print(f'relative importance lactationcat vs DME RF {0.061598/0.529451}')
print(f'relative importance lactationcat vs DME XgBoost {0.054133/0.516186}')


In [0]:
def plot_perm_importance(df, title):

    #rename feature labels
    df['feature'] = df['feature'].replace({'DME_prop_logit': 'DME', 'LactationCat': 'Parity', 'CalvingYear': 'Calving Year', 'CalvingSeason_Spring': 'Spring', 'CalvingSeason_Summer': 'Summer', 'CalvingSeason_Autumn': 'Autumn', 'CalvingSeason_Winter': 'Winter'})

    plt.figure()
    plt.barh(df['feature'], df['importance_mean'])
    plt.xlabel("Permutation Importance (mean ΔR²)")
    plt.title(title)
    plt.gca().invert_yaxis()
    plt.xlim(left=0, right = 1)  # Ensure x-axis starts at 0
    plt.show()



plot_perm_importance(rf_importance, "Random forest permutation importance cumulative yield")
plot_perm_importance(xgb_importance, "XGBoost permutation importance cumulative yield")

no title version of plots

In [0]:
plot_perm_importance(rf_importance, "")
plot_perm_importance(xgb_importance, "")

## SHAP 

In [0]:
import shap
shap.initjs()

print(shap.__version__)

In [0]:
# Small example DataFrame for reproducing XGBoost SHAP error
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

# Example data
example_data = {
    'DME_prop_logit': [0.1, 0.2, 0.3, 0.4],
    'LactationCat': [1, 2, 1, 3],
    'CalvingYear': [2020, 2021, 2020, 2022],
    'CalvingSeason_Spring': [1, 0, 0, 0],
    'CalvingSeason_Summer': [0, 1, 0, 0],
    'CalvingSeason_Autumn': [0, 0, 1, 0],
    'CalvingSeason_Winter': [0, 0, 0, 1]
}
example_target = [100, 150, 120, 130]

X_train = pd.DataFrame(example_data).astype(float)
y_train = pd.Series(example_target).astype(float)

# Fit model
XgBoost = XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators=10, learning_rate=0.1, min_child_weight=1, max_depth=2).fit(X_train, y_train)

explainer = shap.TreeExplainer(XgBoost)
shap_values = explainer(X_train)

### RandomForest

In [0]:
#recreate best fitted models for RF and XgBoost 
#define input and output variables 
X_train = train_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
y_train = train_df['CummulativeYield']

X_train = X_train.rename(columns={'DME_prop_logit': 'DME', 'LactationCat': 'Parity', 'CalvingYear': 'Calving Year', 'CalvingSeason_Spring': 'Spring', 'CalvingSeason_Summer': 'Summer', 'CalvingSeason_Autumn': 'Autumn', 'CalvingSeason_Winter': 'Winter'})

#fit model 
RF = RandomForestRegressor(n_estimators=1000, max_depth=3, min_samples_leaf=5, min_samples_split=2, random_state=42).fit(X_train, y_train)

Combine shap values calvingseason

In [0]:
# get shap values
explainer = shap.Explainer(RF)
shap_values = explainer(X_train)

print(shap_values)

In [0]:
import shap

# get shap values
explainer = shap.Explainer(RF)
shap_values = explainer(X_train)

#sum up season category
calving_season_cols = [
    'Spring',
    'Summer',
    'Autumn',
    'Winter'
]

# indices
season_idx = [shap_values.feature_names.index(col) for col in calving_season_cols]

# summed shap values
calving_season_sum = shap_values.values[:, season_idx].sum(axis=1)

# remaining columns
remaining_idx = [
    i for i, col in enumerate(shap_values.feature_names)
    if col not in calving_season_cols
]

# reduced arrays
new_values = shap_values.values[:, remaining_idx]
new_data = shap_values.data[:, remaining_idx]

# insertion position
insert_pos = remaining_idx.index(
    shap_values.feature_names.index('Calving Year')
) + 1

# insert summed SHAP values
new_values = np.insert(
    new_values,
    insert_pos,
    calving_season_sum,
    axis=1
)

# create numeric category encoding
season_numeric = shap_values.data[:, season_idx].argmax(axis=1)

# insert numeric values into data
new_data = np.insert(
    new_data,
    insert_pos,
    season_numeric,
    axis=1
)

# feature names
new_feature_names = [
    shap_values.feature_names[i]
    for i in remaining_idx
]
new_feature_names.insert(insert_pos, "Calving Season")

# create new explanation
shap_values_new = shap.Explanation(
    values=new_values,
    base_values=shap_values.base_values,
    data=new_data,
    feature_names=new_feature_names
)

#visualize 

print('Random Forest SHAP')
print('Waterfall plots')
# waterfall plot for sample of observation
shap.plots.waterfall(shap_values_new[5220])
shap.plots.waterfall(shap_values_new[0])
shap.plots.waterfall(shap_values_new[5219])
shap.plots.waterfall(shap_values_new[3])

print('')
shap.plots.bar(shap_values_new)
shap.plots.beeswarm(shap_values_new)
shap.plots.scatter(shap_values_new[:, 'DME'], color = shap_values_new)

In [0]:
# waterfall plot for sample of observation
shap.plots.waterfall(shap_values_new[5944])
shap.plots.waterfall(shap_values_new[0])
shap.plots.waterfall(shap_values_new[5942])
shap.plots.waterfall(shap_values_new[3])

In [0]:
shap.plots.bar(shap_values_new)

In [0]:
shap.plots.beeswarm(shap_values_new)

In [0]:
shap.plots.partial_dependence(
    "DME_prop_logit",
    RF.predict,
    X_train,
    model_expected_value=True,
    feature_expected_value=True
)

In [0]:
shap.plots.scatter(shap_values_new[:, 'DME_prop_logit'], color = shap_values)

### XGBoost  

In [0]:
#recreate best fitted models for RF and XgBoost 
#define input and output variables 
# X_train = train_df[['DME_prop_logit', 'LactationCat', 'CalvingYear', 'CalvingSeason_Spring', 'CalvingSeason_Summer', 'CalvingSeason_Autumn', 'CalvingSeason_Winter']]
# y_train = train_df['CummulativeYield']


#fit model 
XgBoost = XGBRegressor(objective='reg:squarederror', random_state=42, n_estimators = 1000, learning_rate = 0.01, min_child_weight = 1, max_depth = 1 ).fit(X_train, y_train)

In [0]:
shap_values_xgboost

In [0]:
explainer_xgboost = shap.Explainer(XgBoost)
shap_values_xgboost = explainer_xgboost(X_train)

In [0]:
%pip install xgboost
from xgboost import XGBRegressor

In [0]:


explainer_xgboost = shap.Explainer(XgBoost)
shap_values_xgboost = explainer_xgboost(X_train)

calving_season_cols = [
    'Spring',
    'Summer',
    'Autumn',
    'Winter'
]

# indices
season_idx = [shap_values_xgboost.feature_names.index(col) for col in calving_season_cols]

# summed shap values
calving_season_sum = shap_values_xgboost.values[:, season_idx].sum(axis=1)

# remaining columns
remaining_idx = [
    i for i, col in enumerate(shap_values_xgboost.feature_names)
    if col not in calving_season_cols
]

# reduced arrays
new_values = shap_values_xgboost.values[:, remaining_idx]
new_data = shap_values_xgboost.data[:, remaining_idx]

# insertion position
insert_pos = remaining_idx.index(
    shap_values_xgboost.feature_names.index('Calving Year')
) + 1

# insert summed SHAP values
new_values = np.insert(
    new_values,
    insert_pos,
    calving_season_sum,
    axis=1
)

# create numeric category encoding
season_numeric = shap_values_xgboost.data[:, season_idx].argmax(axis=1)

# insert numeric values into data
new_data = np.insert(
    new_data,
    insert_pos,
    season_numeric,
    axis=1
)

# feature names
new_feature_names = [
    shap_values_xgboost.feature_names[i]
    for i in remaining_idx
]
new_feature_names.insert(insert_pos, "Calving Season")

# create new explanation
shap_values_xgboost_new = shap.Explanation(
    values=new_values,
    base_values=shap_values_xgboost.base_values,
    data=new_data,
    feature_names=new_feature_names
)

# waterfall plot for sample of observation
print('XGBoost SHAP')
print('Waterfall plots')
shap.plots.waterfall(shap_values_xgboost_new[5220])
shap.plots.waterfall(shap_values_xgboost_new[0])
shap.plots.waterfall(shap_values_xgboost_new[5219])
shap.plots.waterfall(shap_values_xgboost_new[3])

print('')
shap.plots.bar(shap_values_xgboost_new)
shap.plots.beeswarm(shap_values_xgboost_new)
shap.plots.scatter(shap_values_xgboost_new[:, 'DME'], color = shap_values_xgboost_new)


In [0]:
# waterfall plot for sample of observation
shap.plots.waterfall(shap_values_xgboost_new[5944])
shap.plots.waterfall(shap_values_xgboost_new[0])
shap.plots.waterfall(shap_values_xgboost_new[5942])
shap.plots.waterfall(shap_values_xgboost_new[3])

In [0]:
shap.plots.bar(shap_values_xgboost_new)
shap.plots.beeswarm(shap_values_xgboost_new)
shap.plots.scatter(shap_values_xgboost_new[:, 'DME_prop_logit'], color = shap_values_xgboost_new)